# Datos de Giovanni

Carga de los seis CSV originales de `data/giovanni`. Los metadatos se conservan por archivo. Los valores de relleno se representan con `None`; no se convierten unidades.


In [ ]:
import csv
from datetime import datetime
from pathlib import Path

CARPETA_DATOS = Path.cwd() / "data" / "giovanni"


def leer_giovanni(ruta):
    metadatos = {}
    registros = []
    with ruta.open(encoding="utf-8-sig", newline="") as archivo:
        lector = csv.reader(archivo, skipinitialspace=True)
        for fila in lector:
            if not fila:
                continue
            if fila[0].strip().lower() == "time":
                columna = fila[1].strip()
                break
            metadatos[fila[0].strip().rstrip(":")] = ",".join(fila[1:]).strip()
        else:
            raise ValueError(f"No se encontro la cabecera time en {ruta.name}")

        relleno = next(
            (float(valor) for clave, valor in metadatos.items()
             if clave.startswith("Fill Value")),
            None,
        )
        for fila in lector:
            if not fila:
                continue
            valor = float(fila[1]) if fila[1].strip() else None
            registros.append({
                "fecha": datetime.fromisoformat(fila[0].strip()),
                "valor": None if valor == relleno else valor,
            })
    return {"columna_original": columna, "metadatos": metadatos, "registros": registros}


archivos = sorted(CARPETA_DATOS.glob("*.csv"))
if not archivos:
    raise FileNotFoundError(f"No hay CSV en {CARPETA_DATOS}. Ejecuta desde la raiz del proyecto.")
datos = {ruta.stem: leer_giovanni(ruta) for ruta in archivos}
print(f"Series cargadas: {len(datos)}")


In [ ]:
for nombre, serie in datos.items():
    registros = serie["registros"]
    fechas = [registro["fecha"] for registro in registros]
    faltantes = sum(registro["valor"] is None for registro in registros)
    print(nombre)
    print(f"  {len(registros)} registros | {min(fechas):%Y-%m} a {max(fechas):%Y-%m} | {faltantes} faltantes")

# Ejemplo: primeras cinco observaciones de precipitaciones.
datos["Precipitaciones-GPM_3IMERGM-Giovanni"]["registros"][:5]
